# Stretch: the Spotify route

For when the first run is done and there is time left, and for anyone
whose project will use their own Spotify **Extended streaming history**.
The stretches get harder as you go, so nobody is expected to finish them
all. Each one adds one idea.

Your real export takes days to arrive, so this notebook starts by writing
a small **practice export** into `output/`: the same file names and field
names as the real thing, with every value made up. When your real export
arrives, you change one path and run the same code on it.

Where the field names come from: Spotify's own description of the export
(support.spotify.com, *Understanding my data*) lists what each record
contains but not the technical names. The names used here are the ones
that appear in real exports, as published by people who have analysed
their own. **Your export comes with a file called *Read Me First -
Extended Streaming History*. That file is the authority; check these names
against it, and against `df.columns`, before you trust anything.**

Where a stretch has `assert` lines, uncomment them once your code exists.
They are the real numbers for the practice export.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

In [ ]:
# Run this cell as it is. It writes a small PRACTICE export: the field names
# are the real ones, every value is made up. Nothing here is anyone's data.
import json
import random
from datetime import datetime, timedelta, timezone

PRACTICE = Path("output") / "session-11" / "spotify-practice"
PRACTICE.mkdir(parents=True, exist_ok=True)

rng = random.Random(11)
artists = {
    "Glass Tram": ["Broken Lines", "Amber Ferry", "Night Bus"],
    "Sara Lindqvist": ["Salt Lines", "Small Hours", "Harbour"],
    "DJ Kompas": ["Broken Lines", "North Star"],
    "Fjord & Flint": ["Golden Garden", "Summer Season"],
    "Odessa Brass": ["Golden Avenue"],
}
shows = {"The Long Listen": ["Episode 12", "Episode 13"]}
start = datetime(2024, 1, 1, tzinfo=timezone.utc)


def practice_record():
    """One made-up stream, with the field names of a real extended history file."""
    ended = start + timedelta(seconds=rng.randrange(2 * 365 * 24 * 3600))
    is_podcast = rng.random() < 0.08
    played_ms = rng.choice([rng.randrange(1_000, 30_000), rng.randrange(60_000, 330_000)])
    record = {
        "ts": ended.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "platform": rng.choice(["android", "ios", "windows"]),
        "ms_played": played_ms if not is_podcast else rng.randrange(300_000, 3_000_000),
        "conn_country": rng.choice(["NL", "NL", "NL", "DE"]),
        "master_metadata_track_name": None,
        "master_metadata_album_artist_name": None,
        "master_metadata_album_album_name": None,
        "spotify_track_uri": None,
        "episode_name": None,
        "episode_show_name": None,
        "spotify_episode_uri": None,
        "reason_start": rng.choice(["trackdone", "clickrow", "fwdbtn"]),
        "reason_end": rng.choice(["trackdone", "fwdbtn", "endplay"]),
        "shuffle": rng.random() < 0.5,
        "skipped": rng.choice([True, False, False, False, None]),
        "offline": False,
        "offline_timestamp": None,
        "incognito_mode": False,
    }
    if is_podcast:
        show = rng.choice(list(shows))
        record["episode_name"] = rng.choice(shows[show])
        record["episode_show_name"] = show
        record["spotify_episode_uri"] = "spotify:episode:practice"
    else:
        artist = rng.choice(list(artists))
        record["master_metadata_track_name"] = rng.choice(artists[artist])
        record["master_metadata_album_artist_name"] = artist
        record["master_metadata_album_album_name"] = "Practice Album"
        record["spotify_track_uri"] = "spotify:track:practice"
    return record


records = sorted((practice_record() for _ in range(600)), key=lambda r: r["ts"])
files = {
    "Streaming_History_Audio_2024_0.json": records[:350],
    "Streaming_History_Audio_2024-2025_1.json": records[350:],
}
for name, chunk in files.items():
    with open(PRACTICE / name, "w", encoding="utf-8") as f:
        json.dump(chunk, f, indent=1)
    print(f"wrote {PRACTICE / name}  ({len(chunk)} streams)")

## Stretch 1: a JSON file is a list of dictionaries

Open one of the files in VS Code and look at it before writing any code.
Square bracket, then curly brackets, over and over.

That is the shape from session 4: **a list of dictionaries, one per row**.
JSON just writes it down with `true`, `false` and `null` where Python says
`True`, `False` and `None`.

First, read it with the standard library's `json` module, the way session
4 would have: `json.load(f)` inside a `with open(...)`. How many records?
What does the first one look like?

In [ ]:
first_file = PRACTICE / "Streaming_History_Audio_2024_0.json"

# TODO: records = ...


# assert len(records) == 350
# assert records[0]["ts"] == "2024-01-03T14:21:38Z"     # the first stream in the file

Now the pandas way: `pd.read_json(path)` reads exactly that shape into a
DataFrame. Same number of rows? Which columns are not the type you would
expect? (Look at `ts` and `skipped` in particular.)

In [ ]:
# TODO: one = pd.read_json(first_file)


# assert len(one) == len(records)

## Stretch 2: all the files at once

A real export is split into several files of about 12 MB each. You want
one DataFrame.

* `PRACTICE.glob("Streaming_History_Audio_*.json")` finds every file whose
  name matches the pattern. The `*` means "anything here".
* `pd.concat(list_of_dataframes, ignore_index=True)` stacks DataFrames on
  top of each other, with fresh row numbers.

Write `load_streams(folder)` that returns all the audio streams in one
DataFrame. Print the file names in the order `glob` (sorted) gives them.
Is it the order you expected?

In [ ]:
# TODO: def load_streams(folder):


# streams = load_streams(PRACTICE)
# assert len(streams) == 600

**Think about it:** why is it safer to sort the rows by `ts` after loading
than to rely on the order of the files?

*Your answer here.*

## Stretch 3: what time was it, really?

Spotify says `ts` is the date and time the stream **ended**, in UTC.

* `pd.to_datetime(streams["ts"], utc=True)` makes it a real timestamp that
  knows it is UTC
* `.dt.tz_convert("Europe/Amsterdam")` shows the same moment as a clock in
  the Netherlands would

Add a column `local` with the Amsterdam time. Then count how many streams
have a **different date** in UTC than in Amsterdam (`.dt.date` gives the
date part).

In [ ]:
# TODO


# assert changed_date == 35

**Think about it:** for which questions does this matter, and for which
does it not? "Minutes per year"? "What time of day do I listen most"?

*Your answer here.*

## Stretch 4: music or podcast?

The same files hold both. A music stream has a
`master_metadata_track_name`; a podcast stream has an `episode_name`
instead, and the track fields are empty.

Count each kind, and the total minutes of each (`ms_played` is in
milliseconds: divide by 60,000).

In [ ]:
# TODO


# assert (len(music), len(podcasts)) == (556, 44)

Which kind has more minutes? What would have happened to a "minutes of
listening per artist" project that forgot this step?

*Your answer here.*

## Stretch 5: what counts as a play?

Spotify records **every** stream, including the two seconds you heard
before skipping. Our course data only ever had real plays in it. So you
have to decide what a "play" is, and the decision changes the answer.

Find the top artist by number of music streams, first counting
everything, then counting only streams of at least 30 seconds (30,000
ms). Then look at the `skipped` column with `value_counts(dropna=False)`.

In [ ]:
# TODO


# assert top_all == "Glass Tram"
# assert top_30s == "DJ Kompas"

**Think about it:** the practice file exaggerates this on purpose (about
half its music streams are short). But even a real one will move. Which
threshold would you choose for your project, and how would you say so in
session 12? And what will you do with the rows where `skipped` is
missing?

*Your answer here.*

## Stretch 6: into the shape of the course data

The pipeline from session 10 was built for `data/clean/plays.csv`. If the
Spotify streams had the **same column names**, most of it would work on
them unchanged.

Write `spotify_to_plays(streams, min_ms=30_000)` that returns music
streams only, of at least `min_ms`, with these columns:

| Course column | From |
|---|---|
| `played_at` | the Amsterdam date, as `YYYY-MM-DD` text |
| `artist_name` | `master_metadata_album_artist_name` |
| `track_name` | `master_metadata_track_name` |
| `minutes_played` | `ms_played / 60_000`, rounded to 2 decimals |
| `device` | `platform` |
| `skipped` | `skipped`, with missing treated as not skipped, as 0 or 1 |

The last row is a decision. Write down why, or choose differently.

In [ ]:
# TODO: def spotify_to_plays(streams, min_ms=30_000):


# plays = spotify_to_plays(streams)
# course_columns = pd.read_csv("data/clean/plays.csv", nrows=1).columns
# assert set(plays.columns) <= set(course_columns)
# assert len(plays) == 284

Last: the worked example asked which genres grew from 2024 to 2025. Try
that on `plays`. What goes wrong, and what does it tell you about scoping
a Spotify project? Ask the same question **per artist** instead.

In [ ]:
# TODO

*Your answer here.*

## When your real export arrives

1. Unzip it into `project/data/spotify/`. Then read the section on private
   data on the slides **before** your next `git add`.
2. Open the *Read Me First* file and compare its field names with the ones
   used here.
3. Point `load_streams` at the folder, and run `first_look` from notebook
   01 on the result.
4. Expect differences from the practice export: more files, other
   `platform` values, perhaps other files next to the audio ones. The
   `Streaming_History_Audio_*` pattern deliberately picks only the audio.